# Predicting Next-Iteration Eqsat Memory

Given one eqsat iteration's state — egraph size, rule applications, timings,
live heap — how well can classical ML predict the *next* iteration's memory
use?

The data comes from `generate.py`, which records a per-iteration `Measurement`
for every seed term in `data/seed_terms/*/terms.json`. Each iteration carries
egg's own counters plus a jemalloc live-heap reading, so a single seed folder
yields thousands of labelled transitions.

Two things decide whether the numbers below mean anything:

- **Two targets.** Memory is strongly autocorrelated, so predicting the
  absolute next value scores well even when the model has learned nothing —
  "next ≈ current" is already an excellent guess. Predicting the *growth
  ratio* removes that freebie and isolates the real signal. Both are reported.
- **Grouped splits.** Iterations within one run are near-duplicates of their
  neighbours. A random row split would put iteration 40 in train and 41 in
  test, and the score would be fiction. Every split here keeps whole seed
  terms on one side.

In [ ]:
import altair as alt
import polars as pl

import iteration_data as D
import memory_model as M
import memory_plots as MP
import plots as P

alt.theme.register("analysis", enable=True)(lambda: P.THEME)

## Load the iteration traces

`resolve_seed_dir` takes a name fragment and picks the newest match; with no
argument it uses the most recent seed folder. `load_iterations` flattens every
term's trace into one row per iteration, expanding the `applied` map into a
`rule_<name>` column per rewrite.

In [ ]:
SEED_DIR = D.resolve_seed_dir()

iterations = D.load_iterations(SEED_DIR)
iterations.select(
    "term_size", "iter_index", "egraph_nodes", "egraph_classes", "allocated", "n_rebuilds"
).head()

## Build the supervised frame

Each row pairs iteration `i`'s features with iteration `i+1`'s memory. Two
classes of row are dropped:

- **Run ends**, which have no successor to predict.
- **Transitions into a stop iteration.** egg's final iteration reports a
  pre-apply node count alongside a post-run heap reading — one trace ends at
  49k nodes but 3.0 GB — so its memory is not comparable with a mid-run
  iteration and would poison the target.

Derived features (`nodes_per_class`, `bytes_per_node`, `prev_growth`) look
only backwards, so nothing downstream sees the future.

In [ ]:
transitions = D.build_transitions(iterations)

SCALARS, RULES = D.feature_columns(transitions)
FEATURES = SCALARS + RULES
print(f"{len(SCALARS)} scalar features, {len(RULES)} rewrite-rule features")

transitions.select(
    "term_size", "iter_index", "egraph_nodes", "allocated", "next_allocated", "y_log_growth"
).head()

## What the targets look like

The growth ratio is sharply peaked at zero — most iterations barely move the
heap — with a long right tail where the egraph blows up. Those tail events are
the interesting ones, and the reason plain accuracy near the mode is not a
useful summary.

In [ ]:
MP.growth_histogram(transitions)

## Cross-validated model comparison

Three predictors under 5-fold `GroupKFold` on the seed term:

- **naive (carry forward)** — assume memory does not change. The bar to beat.
- **ridge** — linear on log-scaled features.
- **gradient boosting** — `HistGradientBoostingRegressor` on raw features.

`median error ×` converts the log-space error back to a multiplicative factor:
1.03 means the typical prediction is within 3% of the true memory.

In [ ]:
metrics, predictions = M.evaluate(transitions, FEATURES, RULES)
metrics

In [ ]:
MP.metric_bars(metrics, metric="R2")

### Reading the two targets together

The absolute target flatters everyone: carry-forward alone scores R² ≈ 0.95,
because memory rarely moves far in one iteration. Reported on its own, that
number would say almost nothing about the model.

The growth target is the honest one. Carry-forward scores *negative* R² there
— worse than predicting the mean — while gradient boosting reaches ≈ 0.65. The
gap between those two is the part that is actually learned, and the tree model
roughly halves the naive error in log space.

In [ ]:
MP.predicted_vs_actual(predictions, "log memory growth ratio")

In [ ]:
MP.residual_distribution(predictions, "log memory growth ratio")

Residuals against egraph size show whether error is scale-dependent — a model
that is accurate on small egraphs but drifts on large ones is the wrong tool
for predicting a blow-up.

In [ ]:
MP.residual_vs_size(predictions, "log memory growth ratio")

## What the model uses

Permutation importance for the boosted model on the growth target, fitted on
four folds and permuted on the held-out fifth, so it reflects generalisation
to unseen terms rather than in-sample fit.

In [ ]:
importance = M.importances(transitions, FEATURES, RULES, target="y_log_growth")
MP.importance_bars(importance)

Current heap and `search_time` dominate: how much memory is already committed,
and how long matching took, together say most of what there is to say about
the next iteration. Egraph size and `bytes_per_node` follow.

The per-rule counts land well below the scalars. Which rewrites fired last
iteration adds little once size and timing are known — worth noting, since the
opposite would have been the more interesting result.

## Does more history help?

Everything above predicts from a single iteration. But an eqsat run is a
trajectory: memory that has doubled three iterations running is in a different
regime than memory that has been flat, even when the current reading is
identical. That difference is invisible to a one-iteration model.

`build_transitions(..., window=n)` attaches the previous `n - 1` iterations of
each scalar, plus per-step log deltas measuring the move between consecutive
lags. Two details keep the comparison honest:

- **Lags are taken before the row filters.** Some runs have iterations dropped
  mid-trace (a zero heap reading), so lagging the filtered frame would build
  windows that silently span a gap and pair non-adjacent iterations.
- **Warm-up rows are backfilled, not dropped.** A row two iterations into a run
  has no `lag3`; it repeats the run's first iteration, which makes its deltas
  zero — "no change observed yet". Every window size therefore scores the exact
  same 10,826 rows, so the curve below reflects the features and not a shifting
  eval set.

Rule counts stay at the current iteration only. They already rank below the
scalars in permutation importance, and lagging all 36 would quadruple the
feature count to buy the model's weakest signal.

In [ ]:
sweep = M.window_sweep(iterations, windows=(1, 2, 3, 4, 6, 8))
sweep.filter(pl.col("target") == "log memory growth ratio").select(
    "window", "model", "R2", "MAE (log)", "median error x", "n_features"
).sort("model", "window")

In [ ]:
MP.window_sweep_chart(sweep, metric="R2")

## Extrapolating to larger terms

Grouped CV holds out unseen terms, but they are drawn from the same size
range. The harder question: train only on small seed terms (size < 36) and
test on the large ones.

In [ ]:
M.size_extrapolation(transitions, FEATURES, RULES, split_size=36)

## Summary

Next-iteration memory is genuinely predictable from the last iteration alone.
Gradient boosting lands within a few percent of the true value and holds up
when extrapolating from small terms to large ones, so a running eqsat could
use it as a cheap one-step-ahead warning.

Widening the input to several past iterations helps, but only a little: R²
rises from 0.653 to 0.689 at a four-iteration window and saturates there. The
process is close to Markovian in this data — the current egraph state carries
most of the signal, and the trajectory that produced it adds a modest amount
at the margin.

Caveats worth keeping in view:

- **One horizon.** This predicts one iteration ahead, and widening the input
  window does not change that — it is still one step out. Compounding it over
  many steps to forecast a final footprint is a different, harder problem, and
  the one where trajectory features would likely matter more.
- **One language and one seed folder.** All results come from a single
  `data/seed_terms` directory; the rule columns are that language's rewrites.
  Re-run against another folder before assuming any of it transfers.
- **Stop iterations are excluded.** The model never sees the transition into a
  run's final iteration, which is exactly the moment a memory-limit kill
  happens. Predicting *that* would need the stop rows re-included and their
  post-run heap reading handled deliberately.